In [15]:
import zipfile
import os
import pandas as pd
zip_file_path = '/archive.zip'

extraction_dir = '/tmp/air_quality_data'
os.makedirs(extraction_dir, exist_ok=True)

print(f"Extracting {zip_file_path} to {extraction_dir}...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_dir)

print("Extraction complete. Listing contents:")
extracted_files = os.listdir(extraction_dir)
for f in extracted_files:
    print(f"- {f}")

csv_file = None
for f in extracted_files:
    if f.endswith('.csv'):
        csv_file = os.path.join(extraction_dir, f)
        break

if csv_file:
    print(f"Found CSV file: {csv_file}")
else:
    raise FileNotFoundError("No CSV file found in the extracted archive.")

print(f"Loading dataset from {csv_file}...")
df = pd.read_csv(csv_file)

Extracting /archive.zip to /tmp/air_quality_data...
Extraction complete. Listing contents:
- station_day.csv
- city_hour.csv
- city_day.csv
- stations.csv
- station_hour.csv
Found CSV file: /tmp/air_quality_data/station_day.csv
Loading dataset from /tmp/air_quality_data/station_day.csv...


### Display the first 5 rows to understand the data structure

In [16]:
display(df.head())

,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,AP001,2017-11-24,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,0.17,5.92,0.10,NaN,NaN
1,AP001,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,0.20,6.50,0.06,184.0,Moderate
2,AP001,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,0.22,7.95,0.08,197.0,Moderate
3,AP001,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,0.29,7.63,0.12,198.0,Moderate
4,AP001,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,0.17,5.02,0.07,188.0,Moderate


### Check missing values for every column and display DataFrame info

In [18]:
df.info()
print("\nMissing values per column:")
display(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108035 entries, 0 to 108034
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   StationId   108035 non-null  object 
 1   Date        108035 non-null  object 
 2   PM2.5       86410 non-null   float64
 3   PM10        65329 non-null   float64
 4   NO          90929 non-null   float64
 5   NO2         91488 non-null   float64
 6   NOx         92535 non-null   float64
 7   NH3         59930 non-null   float64
 8   CO          95037 non-null   float64
 9   SO2         82831 non-null   float64
 10  O3          82467 non-null   float64
 11  Benzene     76580 non-null   float64
 12  Toluene     69333 non-null   float64
 13  Xylene      22898 non-null   float64
 14  AQI         87025 non-null   float64
 15  AQI_Bucket  87025 non-null   object 
dtypes: float64(13), object(3)
memory usage: 13.2+ MB

Missing values per column:


,0
StationId,0
Date,0
PM2.5,21625
PM10,42706
NO,17106
NO2,16547
NOx,15500
NH3,48105
CO,12998
SO2,25204


### Handle missing values appropriately and explain your choice

In [19]:
initial_rows = df.shape[0]
initial_cols = df.shape[1]
threshold = 0.5 * len(df)
df = df.dropna(axis=1, thresh=threshold)
print(f"Dropped {initial_cols - df.shape[1]} columns with more than 50% missing values.")

for col in df.columns:
    if df[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f"Filled missing values in numeric column '{col}' with median: {median_val}")
        elif pd.api.types.is_object_dtype(df[col]):

            mode_val = df[col].mode()[0]
            df[col].fillna(mode_val, inplace=True)
            print(f"Filled missing values in categorical column '{col}' with mode: {mode_val}")
        else:
            df.dropna(subset=[col], inplace=True)
            print(f"Dropped rows with missing values in column '{col}' of other data types.")

print("\nMissing values after handling:")
display(df.isnull().sum())

Dropped 1 columns with more than 50% missing values.
Filled missing values in numeric column 'PM2.5' with median: 55.95
Filled missing values in numeric column 'PM10' with median: 122.09
Filled missing values in numeric column 'NO' with median: 10.29
Filled missing values in numeric column 'NO2' with median: 27.21
Filled missing values in numeric column 'NOx' with median: 26.66
Filled missing values in numeric column 'NH3' with median: 23.59
Filled missing values in numeric column 'CO' with median: 0.91
Filled missing values in numeric column 'SO2' with median: 8.95
Filled missing values in numeric column 'O3' with median: 30.84
Filled missing values in numeric column 'Benzene' with median: 1.21
Filled missing values in numeric column 'Toluene' with median: 4.33
Filled missing values in numeric column 'AQI' with median: 132.0
Filled missing values in categorical column 'AQI_Bucket' with mode: Moderate

Missing values after handling:


/tmp/ipykernel_11338/3710905108.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)
/tmp/ipykernel_11338/3710905108.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usin

,0
StationId,0
Date,0
PM2.5,0
PM10,0
NO,0
NO2,0
NOx,0
NH3,0
CO,0
SO2,0


### Remove duplicate records

In [ ]:
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Removed {initial_rows - df.shape[0]} duplicate rows.")
print(f"DataFrame now has {df.shape[0]} rows.")

### Clean all text columns (`strip()`, `title()`, `lower()` where appropriate)

In [20]:
for col in df.select_dtypes(include='object').columns:

    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].str.lower()
    print(f"Cleaned text column: {col} (stripped whitespace and converted to lowercase).")
    print("\nFirst 5 rows after cleaning text columns:")
    display(df.head())

Cleaned text column: StationId (stripped whitespace and converted to lowercase).

First 5 rows after cleaning text columns:


,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,AQI,AQI_Bucket
0,ap001,2017-11-24,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,0.17,5.92,132.0,Moderate
1,ap001,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,0.20,6.50,184.0,Moderate
2,ap001,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,0.22,7.95,197.0,Moderate
3,ap001,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,0.29,7.63,198.0,Moderate
4,ap001,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,0.17,5.02,188.0,Moderate


Cleaned text column: Date (stripped whitespace and converted to lowercase).

First 5 rows after cleaning text columns:


,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,AQI,AQI_Bucket
0,ap001,2017-11-24,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,0.17,5.92,132.0,Moderate
1,ap001,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,0.20,6.50,184.0,Moderate
2,ap001,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,0.22,7.95,197.0,Moderate
3,ap001,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,0.29,7.63,198.0,Moderate
4,ap001,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,0.17,5.02,188.0,Moderate


Cleaned text column: AQI_Bucket (stripped whitespace and converted to lowercase).

First 5 rows after cleaning text columns:


,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,AQI,AQI_Bucket
0,ap001,2017-11-24,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,0.17,5.92,132.0,moderate
1,ap001,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,0.20,6.50,184.0,moderate
2,ap001,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,0.22,7.95,197.0,moderate
3,ap001,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,0.29,7.63,198.0,moderate
4,ap001,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,0.17,5.02,188.0,moderate


### Fix incorrect data types

In [27]:
for col in df.columns:
    if 'date' in col.lower() or 'time' in col.lower():
        try:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            if df[col].isnull().any():
                initial_rows_col = df.shape[0]
                df.dropna(subset=[col], inplace=True)
                print(f"Converted '{col}' to datetime and dropped {initial_rows_col - df.shape[0]} rows with unparseable dates.")
            else:
                print(f"Converted '{col}' to datetime.")
        except Exception as e:
            print(f"Could not convert '{col}' to datetime: {e}")
    elif pd.api.types.is_object_dtype(df[col]):
        try:
            converted_col = pd.to_numeric(df[col], errors='coerce')
            if converted_col.notna().sum() / len(df) > 0.8:
                df[col] = converted_col
                if df[col].isnull().any():
                    median_val = df[col].median()
                    df[col].fillna(median_val, inplace=True)
                    print(f"Converted '{col}' to numeric and filled NaNs with median: {median_val}")
                else:
                    print(f"Converted '{col}' to numeric.")
        except Exception:
            pass

print("\nDataFrame info after fixing data types:")
df.info()

Converted 'Date' to datetime.

DataFrame info after fixing data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108035 entries, 0 to 108034
Data columns (total 15 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   StationId   108035 non-null  object        
 1   Date        108035 non-null  datetime64[ns]
 2   PM2.5       108035 non-null  float64       
 3   PM10        108035 non-null  float64       
 4   NO          108035 non-null  float64       
 5   NO2         108035 non-null  float64       
 6   NOx         108035 non-null  float64       
 7   NH3         108035 non-null  float64       
 8   CO          108035 non-null  float64       
 9   SO2         108035 non-null  float64       
 10  O3          108035 non-null  float64       
 11  Benzene     108035 non-null  float64       
 12  Toluene     108035 non-null  float64       
 13  AQI         108035 non-null  float64       
 14  AQI_Bucket  108035 non-null  

### Rename columns where necessary for consistency

*Note: Without specific knowledge of the dataset's columns, this step is a placeholder. You would typically rename columns for clarity or to adhere to a specific naming convention (e.g., snake_case).* If you need to rename columns, uncomment and modify the example below.

In [26]:
print("No columns renamed in this step as no specific renaming instructions were provided.")

No columns renamed in this step as no specific renaming instructions were provided.


### Create at least one new useful column using Python

In [28]:
datetime_cols = df.select_dtypes(include=['datetime64[ns]']).columns

if len(datetime_cols) > 0:
    date_col = datetime_cols[0]
    df['Year'] = df[date_col].dt.year
    df['Month'] = df[date_col].dt.month
    df['DayOfWeek'] = df[date_col].dt.day_name()
    print(f"Created 'Year', 'Month', and 'DayOfWeek' columns from '{date_col}'.")
    print("\nFirst 5 rows with new columns:")
    display(df.head())
else:
    print("No datetime column found to create 'Year', 'Month', or 'DayOfWeek'.")
    print("As an alternative, let's create a 'RowID' column.")
    df['RowID'] = range(1, len(df) + 1)
    print("Created 'RowID' column.")
    print("\nFirst 5 rows with new 'RowID' column:")
    display(df.head())

Created 'Year', 'Month', and 'DayOfWeek' columns from 'Date'.

First 5 rows with new columns:


,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,AQI,AQI_Bucket,Year,Month,DayOfWeek
0,ap001,2017-11-24,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,0.17,5.92,132.0,moderate,2017,11,Friday
1,ap001,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,0.20,6.50,184.0,moderate,2017,11,Saturday
2,ap001,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,0.22,7.95,197.0,moderate,2017,11,Sunday
3,ap001,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,0.29,7.63,198.0,moderate,2017,11,Monday
4,ap001,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,0.17,5.02,188.0,moderate,2017,11,Tuesday


### Save the cleaned dataset as **cleaned_dataset.csv**

In [29]:
output_csv_path = 'cleaned_dataset.csv'
df.to_csv(output_csv_path, index=False)
print(f"Cleaned dataset saved to '{output_csv_path}'")
print(f"Final cleaned DataFrame shape: {df.shape[0]} rows, {df.shape[1]} columns")

Cleaned dataset saved to 'cleaned_dataset.csv'
Final cleaned DataFrame shape: 108035 rows, 18 columns


### Load the Cleaned Dataset

In [30]:
import pandas as pd
df_cleaned = pd.read_csv('cleaned_dataset.csv')
display(df_cleaned.head())

,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,AQI,AQI_Bucket,Year,Month,DayOfWeek
0,ap001,2017-11-24,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,0.17,5.92,132.0,moderate,2017,11,Friday
1,ap001,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,0.20,6.50,184.0,moderate,2017,11,Saturday
2,ap001,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,0.22,7.95,197.0,moderate,2017,11,Sunday
3,ap001,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,0.29,7.63,198.0,moderate,2017,11,Monday
4,ap001,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,0.17,5.02,188.0,moderate,2017,11,Tuesday
